[Gold, Silver & Precious Metals Futures Daily Data](https://www.kaggle.com/datasets/guillemservera/precious-metals-data)


# STAT5291 Project Code


## Data refresh

From the **repository root** (this folder), regenerate raw downloads and processed CSVs:

```bash
python data_pipeline.py
```

Processed outputs are written under `data/data_processed/`; raw downloads under `data/data_raw/`.
Requires `FRED_API_KEY` in a `.env` file for the HMM feature step.


In [ ]:
# Optional: refresh all data from Python (same as CLI above)
# import data_pipeline
# data_pipeline.run_all()


In [ ]:
import pandas as pd
from pathlib import Path

from IPython.display import display

import utility_func

FIGURE_DIR = Path("figure")
DATA_PROCESSED_DIR = utility_func.DATA_PROCESSED_DIR
DATA_RAW_DIR = utility_func.DATA_RAW_DIR
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ASSET_LIST = ["Gold", "Silver", "Copper", "Platinum", "Palladium", "sp500"]


# EDA
## Generate time series plot


In [ ]:
vix_data = pd.read_csv(DATA_RAW_DIR / "vix.csv", parse_dates=["Date"])

utility_func.plot_time_series(
    vix_data,
    date_col="Date",
    value_col="Close",
    title="VIX by Date",
    ylabel="VIX",
    hline=30,
    save_path=FIGURE_DIR / "vix_hline_30.png",
)


## Log return panel


In [ ]:
Ret_data = pd.read_csv(DATA_PROCESSED_DIR / "LogRet_data.csv", parse_dates=["Date"])
Ret_data = Ret_data.set_index("Date")
Ret_data.index = pd.to_datetime(Ret_data.index, dayfirst=True)

utility_func.plot_time_series(
    Ret_data.reset_index(),
    date_col="Date",
    value_col="sp500",
    ylabel="Log return",
    title="S&P 500 log returns (sample)",
    save_path=FIGURE_DIR / "sp500_sample.png",
)


## Descriptive statistics


In [ ]:
stat_df = utility_func.descriptive_stats_by_regime(Ret_data, ASSET_LIST)
display(stat_df)


## Visualization
### Scatterplot by regime


In [ ]:
utility_func.plot_regime_scatter_grid(Ret_data, ASSET_LIST, FIGURE_DIR)


### QQ-plot


In [ ]:
utility_func.plot_qq_by_regime(Ret_data, ASSET_LIST, FIGURE_DIR, layout="regime_rows")


In [ ]:
events_df = pd.read_csv(DATA_RAW_DIR / "global_market_stress_events.csv")
display(events_df.head())


### Rolling correlation


In [ ]:
utility_func.plot_rolling_corr_stress_events(Ret_data, events_df, FIGURE_DIR)


### Maximum drawdown comparison


In [ ]:
utility_func.plot_drawdown_stress_events(Ret_data, events_df, FIGURE_DIR)


# HMM


## Prepare HMM dataset

Features are built by `data_pipeline.py` into `data/data_processed/HMM_features.csv` (FRED DFII10 / T10YIE, gold ratios, log returns aligned to `LogRet_data.csv` dates).


In [ ]:
hmm_path = DATA_PROCESSED_DIR / "HMM_features.csv"
feat_df = pd.read_csv(hmm_path, parse_dates=["Date"])
display(pd.DataFrame({"rows": [feat_df.shape[0]], "cols": [feat_df.shape[1]]}))
display(feat_df.head())
display(feat_df.tail())


## Feature engineering and in-sample fit


In [ ]:
df_model, X, scaler, features = utility_func.prepare_hmm_design_matrix(hmm_path)
model, df_model = utility_func.fit_gaussian_hmm_assign_states(
    df_model, X, features, scaler, random_state=42
)
utility_func.print_hmm_training_summaries(model, scaler, features)
utility_func.plot_hmm_gold_states_scatter(df_model, model)
utility_func.print_discriminative_feature_scores(model, features)


## Model testing
### Walk-forward and economic value


In [ ]:
df_oos = utility_func.walk_forward_hmm_oos(X, df_model, random_state=0)
df_oos = utility_func.merge_oos_with_returns(df_oos, df_model)
utility_func.print_oos_summary(df_oos)
df_oos = utility_func.add_regime_strategy_returns(df_oos)
utility_func.plot_strategy_vs_buyhold(df_oos)
